## Purpose and mathematical definition

This vignette evaluates two **composed predictions**, separately from the
conditional [residual-risk model](../05a_residual_risk/05a_residual_risk.ipynb).
The [hazard and bands vignette](../05_composed_hazard_and_bands/05_composed_hazard_and_bands.ipynb)
explains the pressure interpretation. This page reports the held-out development
protocol, scores and pooled calibration of h and individual xLoss.

At state $s$, let $p$ be pass-choice probability, $d$ the probability of loss
conditional on a canonical one-second carry, and
$f=\sum_j q_j(1-p_j)$ the pass-failure probability averaged over destination
choice. Let $c$ be terminal individual-loss probability conditional on selecting
and successfully completing that carry, with predictors measured at $s$.

$$h(s)=p f+(1-p)d,$$
$$xLoss(s)=h(s)+(1-p)(1-d)c(s).$$

The first target is **failure of the current canonical action**, interpreted as
fragility of team possession. A successful pass is nonterminal for that team
sequence. The second target is **loss by the end of the current individual
possession**: a successful pass ends the passer's control without loss. This project-specific completion follows total
probability and is an **own derivation**. The residual vignette retains the
conditional model's mathematical definition, full feature descriptions,
training results, feature importance and executable use examples.


## Benchmark of the composed individual xLoss

The same five held-out match pairs evaluate the complete formula
$xLoss=h+(1-p)(1-d)c$ at **15,085 canonical decision checkpoints**, with **2,888
terminal individual losses (19.14%)**. Every component uses its matching outer
fold; no production refit weights generate these predictions. No extra
calibration is fitted to h or to composed xLoss.

| Model and target | States | CE | Six-fit null CE | Gain over null | AUC | Quantile ECE |
|---|---:|---:|---:|---:|---:|---:|
| h: current canonical action fails | 15,085 | 0.272014 | 0.319133 | 14.765% | 0.78168 | 0.749% |
| xLoss: terminal individual loss | 15,085 | **0.421925** | 0.489296 | **13.769%** | 0.75645 | **0.930%** |

**The rows predict different targets**. Their null frequencies and CE scales
differ; the two gain percentages do not measure the same task. Pooled metrics
concatenate all five held-out pairs before scoring; each null is fitted on the
six fitting matches of the same quality cohort. Calibration uses ten equal-count
bins on the pooled predictions, not the average of five fold ECEs.

![Pooled calibration of action-level h and terminal xLoss, each against its own observed target.](../../figures/composed_risk_pooled_calibration.png)

There are 7,820 successful carry checkpoints, 456 failed carry checkpoints and
6,809 pass releases (5,800 completed, 1,009 failed). Whole-parent verification is
the same as for residual training: no shot endings or excluded non-OBE loss
parents, and terminal passes must belong to the verified continuous cohort.
There is no 0.5–10s duration restriction. A short failed carry retains a canonical
one-second forecast; short successful tails are not canonical training actions.
Additional descriptive hazard snapshots are outside this CE evaluation.

For a 2.4s possession ending with a pass, the evaluated decision frames are
0, 1 and 2.4s; 2s is an additional descriptive snapshot. If it ends in direct
loss, frames 0, 1 and 2s are evaluated and the last carry is unsuccessful.

Using base c instead gives xLoss CE 0.421439 and ECE 1.072%. This small composite
ablation is reported without changing the prespecified augmented residual model
based on outer-test performance. Using h alone against the **terminal** target
gives CE 0.500633 and ECE 9.443%: that omits the continuation term and is not a
failure of h on its own action-level target.

These are development OOF results on ten previously studied matches. Upstream
calibration and downstream selection share validation matches. Eligibility
verification is retrospective whole-parent quality selection; it narrows the
population. Residual training observes successful carries, while composition
queries their hypothetical continuation at pass and failed-carry frames too.
Neither this target nor this evaluation claims to reproduce a proprietary xLoss.

[Pooled results and cohort sensitivities](../../data/benchmarks/composed_risk/pooled_metrics.csv),
[calibration bins](../../data/benchmarks/composed_risk/calibration_quantile10.csv),
[folds](../../data/benchmarks/composed_risk/metrics_by_fold.csv),
[matches](../../data/benchmarks/composed_risk/metrics_by_match.csv).


## Reproduce the reported aggregate scores

The following example reads the exported held-out aggregates; it does not refit models or recalculate predictions.

In [5]:
from pathlib import Path
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").is_file())
metrics = pd.read_csv(root / "data/benchmarks/composed_risk/pooled_metrics.csv")
selected = metrics[metrics["cohort"].eq("verified_endpoints") & metrics["model"].isin(["h", "xloss_with_action_choice"])]
assert len(selected) == 2 and selected["n"].eq(15085).all()
print(selected[["target", "model", "n", "losses", "cross_entropy",
                "null_cross_entropy", "gain_on_null", "ece_quantile10"]].to_string(index=False))


                  target                    model     n  losses  cross_entropy  null_cross_entropy  gain_on_null  ece_quantile10
        next_action_loss                        h 15085    1465       0.272014            0.319133      0.147646        0.007490
terminal_individual_loss xloss_with_action_choice 15085    2888       0.421925            0.489296      0.137690        0.009301
